# Day 34: Observability and Monitoring
**Logging, quality and latency monitoring, drift detection and feedback loops**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas matplotlib

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas matplotlib
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Instrumenting requests: JSON logs and spans

In [ ]:
import json, time, uuid, random, re
from contextlib import contextmanager
import numpy as np, pandas as pd, matplotlib.pyplot as plt

LOG_FILE = "requests.jsonl"
open(LOG_FILE, "w").close()

def mask(text):
    return re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[email]", text)

def log(**fields):
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps({"ts": time.time(), **fields}) + "\n")

@contextmanager
def span(name, trace):
    t = time.perf_counter()
    yield
    trace.append({"span": name, "ms": round((time.perf_counter() - t) * 1000, 1)})

def handle(question, release="1.8.0"):
    rid, trace = uuid.uuid4().hex[:8], []
    with span("retrieve", trace):
        time.sleep(random.uniform(0.005, 0.015)); docs = ["c7", "c2"]
    with span("generate", trace):
        time.sleep(random.uniform(0.04, 0.08)); answer = llm(f"Context: Leave expires on 31 March.\nQuestion: {question}")
    with span("guardrail", trace):
        time.sleep(0.002); verdict = "pass"
    log(event="answer", request_id=rid, release=release, question=mask(question), retrieved=docs, guardrail=verdict,
        latency_ms=round(sum(s["ms"] for s in trace), 1), trace=trace)
    return rid, trace

random.seed(34)
for q in ["When does my leave expire?", "Email me at mei@example.com about leave", "How many leave days?"]:
    rid, trace = handle(q)
    print(rid, " | ".join(f"{s['span']} {s['ms']} ms" for s in trace))
print("\nA logged line:\n", open(LOG_FILE).readlines()[1])

## 3. A simulated month of production traffic
Planted events: a new release on 18 Dec, a latency incident on 24 Dec, and a new question topic growing in the last week.

In [ ]:
rng = np.random.default_rng(34)
days = pd.date_range("2026-12-01", "2026-12-30")
rows = []
for d in days:
    n = int(rng.normal(900, 80))
    release = "1.9.0" if d >= pd.Timestamp("2026-12-18") else "1.8.0"
    visa_share = 0.0 if d < pd.Timestamp("2026-12-23") else min(0.25, 0.03 * (d.day - 22))
    topics = rng.choice(["leave", "it", "travel", "visa"], n, p=np.array([0.4, 0.3, 0.3 - visa_share, visa_share]) / 1.0)
    tok_out = rng.normal(260 if release == "1.9.0" else 160, 30, n)
    lat = rng.gamma(4, 450, n) + tok_out * 4 + (2500 if d == pd.Timestamp("2026-12-24") else 0)
    err = rng.random(n) < (0.06 if d == pd.Timestamp("2026-12-24") else 0.008)
    idk = np.where(topics == "visa", rng.random(n) < 0.6, rng.random(n) < 0.05)
    down_p = np.where(idk, 0.5, 0.18 if release == "1.9.0" else 0.05)
    rows.append(pd.DataFrame({"day": d, "release": release, "topic": topics, "tokens_out": tok_out.round(),
                              "latency_ms": lat.round(), "error": err, "idk": idk, "thumbs_down": rng.random(n) < down_p}))
traffic = pd.concat(rows, ignore_index=True)
traffic["cost"] = (900 * 2.5 + traffic.tokens_out * 10) / 1e6        # illustrative prices only
print(len(traffic), "requests over", traffic.day.nunique(), "days")

## 4. The monitoring dashboard

In [ ]:
daily = traffic.groupby("day").agg(requests=("error", "size"), p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
                                   error_rate=("error", "mean"), cost=("cost", "sum"), thumbs_down=("thumbs_down", "mean"),
                                   tokens_out=("tokens_out", "mean"), idk_rate=("idk", "mean"))
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col, color in zip(axes.flat, ["requests", "p95_ms", "error_rate", "cost", "thumbs_down", "tokens_out"],
                          ["#0FA3A3", "#0FA3A3", "#FF8A3D", "#0B5D63", "#FF8A3D", "#0B5D63"]):
    daily[col].plot(ax=ax, color=color, marker=".", title=col); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 5. Alert rules: sustained and relative

In [ ]:
baseline = daily.rolling(7, min_periods=5).median().shift(1)          # trailing 7-day baseline
rules = {
    "error_spike":    daily.error_rate > 0.05,
    "slow_fixed_4s (noisy)": daily.p95_ms > 4000,                        # a fixed number that ignores normal levels
    "slow_vs_baseline": daily.p95_ms > baseline.p95_ms * 1.3,
    "quality_drop":   daily.thumbs_down > baseline.thumbs_down * 2,
    "idk_rising":     (daily.idk_rate > baseline.idk_rate * 1.5).rolling(2).sum() == 2,   # sustained for 2 days
}
for name, fired in rules.items():
    hits = [d.strftime("%m-%d") for d in daily.index[fired.fillna(False).astype(bool)]]
    print(f"{name:<22} fired on {len(hits):>2} days: {hits[:8]}{' ...' if len(hits) > 8 else ''}")
print("\nThe fixed 4 s rule fires almost daily because normal p95 is already about 4 s: noise people learn to ignore.",
      "Relative rules fire when behaviour changes, then go quiet as the baseline adapts, so act on the first alert.")

### Root cause: compare by release, not only by day

In [ ]:
by_release = traffic.groupby("release").agg(tokens_out=("tokens_out", "mean"), p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
                                            thumbs_down=("thumbs_down", "mean")).round(3)
print(by_release)
print("\nRelease 1.9.0 increased output tokens and thumbs-down rate together: likely a prompt or model change making answers longer and less useful.")

## 6. Input drift: topic mix by week

In [ ]:
traffic["week"] = ((traffic.day - traffic.day.min()).dt.days // 7 + 1).clip(upper=4)
mix = pd.crosstab(traffic.week, traffic.topic, normalize="index").round(2)
print(mix)
print("\nI don't know rate by topic (week 4):\n", traffic[traffic.week == 4].groupby("topic").idk.mean().round(2).to_string())

## 7. Feedback loop: thumbs-down to review queue to golden set

In [ ]:
queue = traffic[traffic.thumbs_down & (traffic.week == 4)].sample(frac=1, random_state=1)
print("Review queue size (week 4):", len(queue))
reviewed = queue.head(5).assign(label=["missing_doc", "missing_doc", "too_long", "missing_doc", "too_long"])
golden_new = [{"id": f"fb-{i + 1:03d}", "question": f"<real {r.topic} question from log>", "must_include": ["<expected phrase>"],
               "source": "user feedback", "failure": r.label, "owner": "mei", "added": "2026-12-30"} for i, r in enumerate(reviewed.itertuples())]
with open("golden_v5_additions.jsonl", "w") as f:
    for g in golden_new:
        f.write(json.dumps(g) + "\n")
print(pd.Series([g["failure"] for g in golden_new]).value_counts().to_string())
print("Added", len(golden_new), "items to golden_v5_additions.jsonl")

### TODO 1: Add a `cost_runaway` alert that fires when daily cost exceeds 1.5x the trailing median, and check whether release 1.9.0 triggers it.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)